# Lab 21 — Cost attribution and adaptive sampling

Two halves. Half A instruments a multi-step agent with OTel baggage carrying tenant.id, user.id, and task.id; demonstrates cross-span propagation; tracks the four token layers; rolls up cost per attribution dimension. Half B configures the Collector with cost-driven sampling policies and implements the external control-loop pattern that adjusts sampling rates based on per-tenant burn rates.

Real OTel SDK runs locally; trace output is visible via ConsoleSpanExporter. The cost-driven sampling policies are demonstrated by simulating the policy logic in Python — production deployment targets a real Collector, which is out of scope for a notebook.

> 📖 Required reading: [`concepts/evaluation/cost-attribution.md`](../../concepts/evaluation/cost-attribution.md), [`concepts/evaluation/adaptive-sampling.md`](../../concepts/evaluation/adaptive-sampling.md).
> ⬅️ Helpful: [Lab 18](../18-opentelemetry-portable-tracing/), [Lab 19](../19-online-evaluation-and-sampling/).
> 🛠 No API keys required; all local.
> ⏱ Run time: 80-100 min including reading.


## Step 0: Setup

OTel SDK with ConsoleSpanExporter so spans print to stdout where the reader can see them. Deterministic seed for synthetic cost data.

In [ ]:
import sys
import os
import json
import yaml
import random
from contextlib import contextmanager

from opentelemetry import baggage, context, trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, BatchSpanProcessor, SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource

# Deterministic
random.seed(42)

# Configure OTel — ConsoleSpanExporter prints to stdout so the reader sees spans inline.
# Production would export to OTLP toward the Collector; same SDK API, different exporter.
resource = Resource.create({"service.name": "lab-21-agent"})
provider = TracerProvider(resource=resource)
# SimpleSpanProcessor (not Batch) so output is synchronous and visible per-cell.
# In production use BatchSpanProcessor for performance.
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("lab-21")
print("OTel tracer ready: service.name=lab-21-agent")
print("  Exporter: ConsoleSpanExporter (spans print to stdout)")
print("  Processor: SimpleSpanProcessor (synchronous, for visibility)")


## ── Half A: Cost attribution via OTel baggage ──

### Step 1: The three attribution dimensions

Three views, each answering a different product question:

| Dimension | Product question | Example signal |
|---|---|---|
| **Per-tenant** | Who's burning the budget? | "Acme Corp = 80% of spend this week" |
| **Per-user** | Which users are heavy? | "5 users burning more than their tier allows" |
| **Per-task** | Which workflow is expensive? | "Document summarization is 12x more expensive than chat" |

You need all three. Building one and retrofitting the others later costs ~5x what building all three up front does — trace identity has to be set at request creation time and propagated through every downstream span. Retroactive log-tagging misses tool spans, partial traces, and timeout edge cases.

This lab implements all three via OTel baggage.

### Step 2: The four token layers

Aggregating "input" and "output" tokens hides where spend goes. The four-layer breakdown that surfaces optimization targets:

| Layer | Attribute | What it counts |
|---|---|---|
| Prompt | `gen_ai.usage.prompt_tokens` | System prompts, few-shot examples, persistent content |
| Tool | `gen_ai.usage.tool_tokens` | Tool descriptions in the schema + tool responses fed back |
| Memory | `gen_ai.usage.memory_tokens` | Retrieved context, RAG documents, conversation history |
| Response | `gen_ai.usage.completion_tokens` | Model output + reasoning tokens |

Each layer has different optimization levers. The lab tracks them separately so the cost rollups can surface which layer is dominating.

In [ ]:
# Token rates per million tokens (USD) - synthetic for demo
# Real 2026 rates: ~$3-15 input, ~$10-75 output per million for frontier models
RATES_PER_M = {
    "prompt":     3.00,   # input-side
    "tool":       3.00,   # input-side (tool descriptions feed in as input)
    "memory":     3.00,   # input-side (retrieved context)
    "completion": 15.00,  # output-side
}

def cost_usd(tokens_by_layer: dict) -> float:
    """Compute total cost across all four token layers."""
    return sum(
        (tokens_by_layer.get(layer, 0) / 1_000_000) * rate
        for layer, rate in RATES_PER_M.items()
    )

# Demonstrate
example = {"prompt": 800, "tool": 200, "memory": 1500, "completion": 400}
print(f"Example token breakdown: {example}")
print(f"  prompt:     ${(example['prompt'] / 1e6) * RATES_PER_M['prompt']:.6f}")
print(f"  tool:       ${(example['tool'] / 1e6) * RATES_PER_M['tool']:.6f}")
print(f"  memory:     ${(example['memory'] / 1e6) * RATES_PER_M['memory']:.6f}")
print(f"  completion: ${(example['completion'] / 1e6) * RATES_PER_M['completion']:.6f}")
print(f"  total:      ${cost_usd(example):.6f}")


### Step 3: The baggage primitive

`baggage.set_baggage(key, value)` creates a new context with that key/value pair. `context.attach(ctx)` makes the context active for the current thread/coroutine. `baggage.get_baggage(key)` reads from the active context.

The key property: baggage flows automatically across function boundaries via the implicit OTel context. You don't pass it as a function argument.

In [ ]:
def inner_function():
    """Called by outer_function. Reads baggage without receiving it as an argument."""
    tenant = baggage.get_baggage("tenant.id")
    user = baggage.get_baggage("user.id")
    print(f"  inner_function sees: tenant.id={tenant}, user.id={user}")

def outer_function():
    """Sets baggage at entry; calls inner without passing tenant/user as args."""
    ctx = baggage.set_baggage("tenant.id", "acme-corp")
    ctx = baggage.set_baggage("user.id", "u-12345", context=ctx)

    print("outer_function set baggage and is about to call inner_function...")
    token = context.attach(ctx)
    try:
        inner_function()  # no arguments — baggage flows via context
    finally:
        context.detach(token)

    # After detach, baggage is no longer visible
    print(f"  after detach: tenant.id={baggage.get_baggage('tenant.id')}")

outer_function()


Notice: `inner_function` reads tenant.id and user.id without `outer_function` passing them as arguments. The values flow via the implicit OTel context. After `context.detach(token)`, the baggage is no longer visible — context is scoped, not global.

This is the entire mechanism. Everything else in Half A is just applying it across a realistic agent call tree.

### Step 4: Multi-step agent simulation

A realistic agent has multiple spans: planner → tool-caller → synthesizer. Baggage set at entry propagates to all three. Each span reads tenant.id / user.id / task.id from baggage and copies them to span attributes.

The redundant `span.set_attribute("tenant.id", ...)` is critical — baggage is the propagation mechanism, but the trace store queries span attributes. Both are required.

In [ ]:
@contextmanager
def request_baggage(tenant_id, user_id, task_id):
    """Context manager that sets baggage at request entry."""
    ctx = baggage.set_baggage("tenant.id", tenant_id)
    ctx = baggage.set_baggage("user.id", user_id, context=ctx)
    ctx = baggage.set_baggage("task.id", task_id, context=ctx)
    ctx = baggage.set_baggage("tenant.tier", tier_for(tenant_id), context=ctx)
    token = context.attach(ctx)
    try:
        yield
    finally:
        context.detach(token)


def tier_for(tenant_id):
    """Synthetic tier lookup."""
    tiers = {"acme-corp": "enterprise", "beta-startup": "premium"}
    return tiers.get(tenant_id, "standard")


def _add_identity_attributes(span):
    """Copy baggage values to span attributes for searchability in the trace store."""
    for key in ("tenant.id", "user.id", "task.id", "tenant.tier"):
        value = baggage.get_baggage(key)
        if value:
            span.set_attribute(key, value)


def planner(query):
    with tracer.start_as_current_span("agent.planner") as span:
        _add_identity_attributes(span)

        # Synthetic token counts for this step
        tokens = {"prompt": 800, "tool": 200, "memory": 0, "completion": 150}
        for layer, count in tokens.items():
            span.set_attribute(f"gen_ai.usage.{layer}_tokens", count)
        span.set_attribute("gen_ai.cost.total_usd", cost_usd(tokens))
        span.set_attribute("gen_ai.operation.name", "plan")

        return ["search_web", "summarize"]


def tool_call(tool_name):
    with tracer.start_as_current_span(f"agent.tool.{tool_name}") as span:
        _add_identity_attributes(span)
        tokens = {"prompt": 600, "tool": 1500, "memory": 0, "completion": 50}
        for layer, count in tokens.items():
            span.set_attribute(f"gen_ai.usage.{layer}_tokens", count)
        span.set_attribute("gen_ai.cost.total_usd", cost_usd(tokens))
        span.set_attribute("gen_ai.operation.name", "tool_call")
        span.set_attribute("tool.name", tool_name)


def synthesizer(query):
    with tracer.start_as_current_span("agent.synthesizer") as span:
        _add_identity_attributes(span)
        tokens = {"prompt": 500, "tool": 0, "memory": 1200, "completion": 350}
        for layer, count in tokens.items():
            span.set_attribute(f"gen_ai.usage.{layer}_tokens", count)
        span.set_attribute("gen_ai.cost.total_usd", cost_usd(tokens))
        span.set_attribute("gen_ai.operation.name", "synthesize")


def run_agent(query, tenant_id, user_id, task_id):
    """Top-level entry — sets baggage, runs the agent pipeline."""
    with request_baggage(tenant_id, user_id, task_id), tracer.start_as_current_span("agent.request") as root:
        _add_identity_attributes(root)
        tools = planner(query)
        for tool in tools:
            tool_call(tool)
        synthesizer(query)


# Run one full request — watch for tenant.id / user.id / task.id on every span
print("Running one full agent request:\n")
run_agent("Summarize 2026 AI agent observability trends", "acme-corp", "u-12345", "summarization")


Look at the spans printed above. Every span — `agent.planner`, `agent.tool.search_web`, `agent.tool.summarize`, `agent.synthesizer`, `agent.request` — has the same `tenant.id`, `user.id`, `task.id`, and `tenant.tier` attributes, even though no function in the chain received those values as arguments. That's the baggage propagation working.

If you'd written this without baggage, every function signature would carry `tenant_id, user_id, task_id` as parameters, and you'd lose them at the first boundary you forgot to pipe them through. With baggage, the gateway middleware sets them once and every downstream span sees them.

### Step 5: Per-span cost rollup

Each span has its four token-layer attributes + a derived `gen_ai.cost.total_usd`. To roll up a trace's total cost, sum across spans. To roll up per-dimension (tenant/user/task), group spans by the identity attribute.

In production, the trace store handles the grouping (SQL query). The lab simulates it with a Python aggregation over a list of synthetic spans.

### Step 6: Burn-down report

200 synthetic traces across 5 tenants and 20 users with different traffic patterns. The "one tenant burning 80%" signal is the one cost attribution catches and dashboards surface.

In [ ]:
TENANTS = [
    ("acme-corp",     "enterprise", 60),   # heavy enterprise — 60% of traffic
    ("beta-startup",  "premium",    20),
    ("gamma-co",      "standard",   10),
    ("delta-inc",     "standard",    7),
    ("epsilon-llc",   "standard",    3),
]
TASKS = ["chat", "summarization", "search", "analysis", "translation"]
TASK_COST_MULT = {                  # tasks have different cost profiles
    "chat":          1.0,
    "summarization": 3.5,            # heavier on memory tokens
    "search":        2.0,
    "analysis":      4.2,            # heaviest — long context + reasoning
    "translation":   1.2,
}


def synthesize_trace_cost(tenant_id, tier, task):
    """Generate a synthetic trace's per-layer tokens and total cost."""
    base = {"prompt": 800, "tool": 200, "memory": 1500, "completion": 400}
    # Scale by task type
    mult = TASK_COST_MULT[task]
    tokens = {layer: int(count * mult * random.uniform(0.7, 1.3)) for layer, count in base.items()}
    cost = cost_usd(tokens)
    return tokens, cost


# Generate 200 traces
N_TRACES = 200
traces = []
for _ in range(N_TRACES):
    # Weighted tenant selection
    tenant_id, tier, weight = random.choices(
        TENANTS, weights=[t[2] for t in TENANTS], k=1
    )[0]
    user_id = f"u-{random.randint(1, 20):03d}"
    task = random.choice(TASKS)
    tokens, cost = synthesize_trace_cost(tenant_id, tier, task)
    traces.append({
        "tenant.id": tenant_id,
        "tenant.tier": tier,
        "user.id": user_id,
        "task.id": task,
        "tokens": tokens,
        "cost_usd": cost,
    })

total = sum(t["cost_usd"] for t in traces)
print(f"Generated {N_TRACES} synthetic traces.  Total cost: ${total:.4f}")
print()

# Per-tenant rollup
print("Per-tenant burn-down (the canonical view):")
print(f"  {'Tenant':<15} {'Tier':<12} {'Traces':>8} {'Cost USD':>12} {'% of total':>12}")
print("  " + "-" * 65)
tenant_totals = {}
for t in traces:
    tenant_totals.setdefault(t["tenant.id"], {"tier": t["tenant.tier"], "count": 0, "cost": 0.0})
    tenant_totals[t["tenant.id"]]["count"] += 1
    tenant_totals[t["tenant.id"]]["cost"] += t["cost_usd"]

for tenant_id, agg in sorted(tenant_totals.items(), key=lambda x: -x[1]["cost"]):
    pct = (agg["cost"] / total) * 100
    print(f"  {tenant_id:<15} {agg['tier']:<12} {agg['count']:>8} {agg['cost']:>12.4f} {pct:>11.1f}%")


In [ ]:
# Per-task rollup — engineering optimization signal
print("Per-task burn-down (engineering optimization signal):")
print(f"  {'Task':<18} {'Traces':>8} {'Cost USD':>12} {'Avg per trace':>15}")
print("  " + "-" * 60)
task_totals = {}
for t in traces:
    task_totals.setdefault(t["task.id"], {"count": 0, "cost": 0.0})
    task_totals[t["task.id"]]["count"] += 1
    task_totals[t["task.id"]]["cost"] += t["cost_usd"]

for task, agg in sorted(task_totals.items(), key=lambda x: -x[1]["cost"]):
    avg = agg["cost"] / agg["count"]
    print(f"  {task:<18} {agg['count']:>8} {agg['cost']:>12.4f} {avg:>15.6f}")

print()
print("Heaviest task: 'analysis' — long context + reasoning tokens dominate.")
print("Targeted optimization (e.g., context compression) on this task gets the most leverage.")


### Step 7: Baggage discipline — what NOT to put in baggage

Four rules:

1. **IDs only, not full objects.** Baggage has a 4KB total size limit (W3C spec). Use `tenant.id="acme-corp"`, not the full tenant record.
2. **No PII.** Baggage propagates over the wire via the `baggage:` HTTP header. Email, name, account number don't belong here.
3. **No secrets.** Tokens, API keys, session credentials don't belong in baggage. Same reason.
4. **Allowlist at ingest.** Configure the Collector to drop unknown baggage keys. External callers can set arbitrary baggage; you don't want their stuff leaking into your spans.

In [ ]:
# Demonstrate the 4KB limit warning
SAFE_BAGGAGE = {
    "tenant.id": "acme-corp",
    "user.id": "u-12345",
    "task.id": "summarization",
    "tenant.tier": "enterprise",
    "request.id": "req-9f8a3b2c-1d4e",
}

UNSAFE_BAGGAGE = {
    "tenant.id": "acme-corp",
    "user.email": "alice@example.com",        # PII — wrong
    "session.token": "sk_live_abc...xyz",     # secret — wrong
    "user.profile": json.dumps({              # full object — wrong
        "name": "Alice", "address": "123 Main St",
        "preferences": [{"theme": "dark"}] * 50,  # bloat — wrong
    }),
}

def baggage_size_bytes(bag):
    """Estimate baggage serialization size in bytes."""
    return sum(len(k.encode()) + len(str(v).encode()) for k, v in bag.items())

print(f"Safe baggage size:   {baggage_size_bytes(SAFE_BAGGAGE):>5} bytes  ({'OK' if baggage_size_bytes(SAFE_BAGGAGE) < 4096 else 'EXCEEDS 4KB'})")
print(f"Unsafe baggage size: {baggage_size_bytes(UNSAFE_BAGGAGE):>5} bytes  ({'OK' if baggage_size_bytes(UNSAFE_BAGGAGE) < 4096 else 'EXCEEDS 4KB'})")
print()
print("The unsafe version also has PII (user.email) and a secret (session.token)")
print("that violate baggage discipline regardless of size.")
print()
print("Allowlist pattern at the Collector ingest: drop any baggage key not in the allowlist.")
print("This protects against external callers injecting arbitrary keys.")


## ── Half B: Adaptive sampling tied to cost ──

Now use the cost attributes from Half A to drive sampling decisions. The Collector's `tailsamplingprocessor` supports composite policies; we'll configure cost-driven retention via `numeric_attribute` and tier-aware retention via `string_attribute`.

### Step 8: Production-realistic Collector YAML

A canonical priority stack for an agent at production scale: errors → latency → high-cost → enterprise-tier → probabilistic-baseline. Evaluation is first-match-wins.

In [ ]:
COLLECTOR_CONFIG_YAML = """
receivers:
  otlp:
    protocols:
      grpc:
        endpoint: 0.0.0.0:4317

processors:
  batch:
    timeout: 1s
    send_batch_size: 1024

  tail_sampling:
    # Wait long enough for agents with multi-second tool calls
    decision_wait: 30s
    # Buffer: 10K traces/sec * 30s * 1.2 safety margin
    num_traces: 360000
    expected_new_traces_per_sec: 10000

    policies:
      # Priority 1: errors (debugging signal)
      - name: errors
        type: status_code
        status_code:
          status_codes: [ERROR]

      # Priority 2: latency outliers (perf signal)
      - name: slow-traces
        type: latency
        latency:
          threshold_ms: 5000

      # Priority 3: high-cost traces (cost-eng signal)
      - name: high-cost
        type: numeric_attribute
        numeric_attribute:
          key: gen_ai.cost.total_usd
          min_value: 0.10

      # Priority 4: enterprise tenants (business value)
      - name: enterprise
        type: string_attribute
        string_attribute:
          key: tenant.tier
          values: [enterprise]

      # Priority 5: probabilistic baseline (statistical floor)
      - name: probabilistic-baseline
        type: probabilistic
        probabilistic:
          sampling_percentage: 5

exporters:
  otlp:
    endpoint: backend:4317
    tls:
      insecure: true

service:
  pipelines:
    traces:
      receivers: [otlp]
      processors: [tail_sampling, batch]
      exporters: [otlp]
"""

# Parse and inspect
config = yaml.safe_load(COLLECTOR_CONFIG_YAML)
policies = config["processors"]["tail_sampling"]["policies"]
print("Tail sampling policies (first-match-wins order):")
for i, p in enumerate(policies, 1):
    print(f"  {i}. {p['name']:<25} (type: {p['type']})")
print()
print(f"decision_wait: {config['processors']['tail_sampling']['decision_wait']}")
print(f"num_traces:    {config['processors']['tail_sampling']['num_traces']:,}")


### Step 9: Simulate the cost-driven policy

The `numeric_attribute` policy on `gen_ai.cost.total_usd` with `min_value: 0.10` keeps every trace whose individual cost exceeds 10 cents. These are exactly the runs you want full visibility on for cost engineering.

We simulate the policy evaluation on the 200 synthetic traces from Step 6.

In [ ]:
def simulate_policies(traces, policies):
    """Simulate first-match-wins policy evaluation. Returns (kept_count, dropped_count)."""
    kept_by_reason = {}
    dropped = 0

    for tr in traces:
        matched = False
        for policy in policies:
            if policy["type"] == "status_code":
                # No error in our synthetic traces — would match here in production
                continue
            elif policy["type"] == "latency":
                # No latency in our cost-focused traces
                continue
            elif policy["type"] == "numeric_attribute":
                key = policy["numeric_attribute"]["key"]
                min_val = policy["numeric_attribute"]["min_value"]
                # Map our trace's cost_usd to the policy's attribute key
                actual = tr.get("cost_usd", 0) if key == "gen_ai.cost.total_usd" else 0
                if actual >= min_val:
                    kept_by_reason[policy["name"]] = kept_by_reason.get(policy["name"], 0) + 1
                    matched = True
                    break
            elif policy["type"] == "string_attribute":
                key = policy["string_attribute"]["key"]
                values = policy["string_attribute"]["values"]
                if tr.get(key) in values:
                    kept_by_reason[policy["name"]] = kept_by_reason.get(policy["name"], 0) + 1
                    matched = True
                    break
            elif policy["type"] == "probabilistic":
                pct = policy["probabilistic"]["sampling_percentage"]
                if random.random() * 100 < pct:
                    kept_by_reason[policy["name"]] = kept_by_reason.get(policy["name"], 0) + 1
                    matched = True
                    break
        if not matched:
            dropped += 1

    return kept_by_reason, dropped


kept_by_reason, dropped = simulate_policies(traces, policies)

print(f"Policy simulation on {N_TRACES} synthetic traces:")
print(f"  {'Reason':<30} {'Kept':>8}")
print("  " + "-" * 42)
for reason, count in sorted(kept_by_reason.items(), key=lambda x: -x[1]):
    print(f"  {reason:<30} {count:>8}")
print(f"  {'(dropped)':<30} {dropped:>8}")
print()
total_kept = sum(kept_by_reason.values())
print(f"Total retained: {total_kept}/{N_TRACES} = {total_kept/N_TRACES*100:.1f}%")
print()
print("Note: 'high-cost' and 'enterprise' policies catch overlapping traces, but first-match-wins")
print("means each trace is attributed to whichever policy matches first in the list.")


### Step 10: The external control loop

The Collector reads static YAML. Adaptive sampling means an external controller polls per-tenant cost metrics and pushes config updates.

Strategy: sampling rate inversely proportional to remaining budget. Tenants with budget headroom get the configured rate; tenants approaching their cap get reduced rates so the remaining observability budget stretches further.

In [ ]:
class AdaptiveSamplingController:
    """External controller that adjusts sampling parameters based on per-tenant burn rate.

    Polls per-tenant monthly spend, computes new sampling parameters, emits updated YAML.
    In production: file-watching, OPAMP push, or remote-config endpoint.
    """

    def __init__(self, base_config: dict, monthly_caps: dict):
        self.base_config = base_config
        self.monthly_caps = monthly_caps  # tenant_id -> cap_usd

    @staticmethod
    def _compute_sampling_rate(remaining_pct: float) -> float:
        """Quadratic falloff. 50% remaining → full rate; 10% remaining → near floor."""
        # rate = (remaining_pct)^2, clamped [0.01, 0.10]
        return max(0.01, min(0.10, remaining_pct ** 2))

    def update_for_burn(self, tenant_spend: dict) -> dict:
        """Compute new sampling rates per tenant based on current spend vs cap."""
        tenant_rates = {}
        for tenant_id, cap in self.monthly_caps.items():
            spend = tenant_spend.get(tenant_id, 0.0)
            remaining = max(0, (cap - spend) / cap)
            tenant_rates[tenant_id] = self._compute_sampling_rate(remaining)
        return tenant_rates

    def emit_updated_config(self, tenant_rates: dict) -> str:
        """In production this would write the YAML to disk or push via OPAMP.
        For the lab, we just return the updated config as a string."""
        # In a real implementation, the controller might emit one string_attribute
        # policy per tenant with that tenant's adjusted probabilistic rate.
        # For pedagogical visibility we just report the rate table.
        return tenant_rates


# Set monthly caps
monthly_caps = {
    "acme-corp":    50.00,   # enterprise: $50/mo budget
    "beta-startup": 20.00,   # premium: $20/mo
    "gamma-co":      5.00,   # standard: $5/mo
    "delta-inc":     5.00,
    "epsilon-llc":   5.00,
}

# Current month's spend (from Step 6 rollup, scaled up to monthly-ish)
current_spend = {t_id: agg["cost"] * 100 for t_id, agg in tenant_totals.items()}

controller = AdaptiveSamplingController(config, monthly_caps)
new_rates = controller.update_for_burn(current_spend)

print(f"{'Tenant':<15} {'Spend USD':>12} {'Cap USD':>10} {'Remaining %':>13} {'New rate':>10}")
print("-" * 70)
for tenant_id in monthly_caps:
    spend = current_spend.get(tenant_id, 0.0)
    cap = monthly_caps[tenant_id]
    remaining_pct = max(0, (cap - spend) / cap) * 100
    rate = new_rates[tenant_id]
    flag = " ← approaching cap, sampling reduced" if rate < 0.05 else ""
    print(f"{tenant_id:<15} {spend:>12.4f} {cap:>10.2f} {remaining_pct:>12.1f}% {rate:>10.3f}{flag}")


Tenants approaching their cap get reduced sampling rates automatically. The probabilistic baseline tightens for the cost-pressured tenant; priority policies (errors, high-cost) still fire at 100% for that tenant — those are the traces you most need when budget is tight.

In production, the controller would:
1. Poll the trace store every N minutes for per-tenant aggregated cost.
2. Compute the new rate table.
3. Either (a) write a new Collector YAML to disk (file-watching reload), (b) push via OPAMP (the OTel control-plane protocol, GA in 2026), or (c) call a remote-config endpoint on the Collector.

The control loop itself is a small service or sidecar. The math is what the lab shows.

### Step 11: The two-tier Collector topology

A constraint at scale: all spans of a trace must reach the same Collector instance for tail sampling to make correct decisions. Tail policies inspect attributes across all spans; if half the spans went to Collector A and half to Collector B, neither has the complete view.

The fix is two tiers. The first tier runs `loadbalancingexporter` which hashes by trace_id; the second tier runs `tailsamplingprocessor` on its sticky subset.

In [ ]:
TIER_1_CONFIG = """
# Tier 1 — load balancing layer
receivers:
  otlp:
    protocols:
      grpc:
        endpoint: 0.0.0.0:4317

exporters:
  loadbalancing:
    # Hash by trace_id so all spans of a trace go to the same tier-2 Collector
    routing_key: traceID
    protocol:
      otlp:
        tls:
          insecure: true
    resolver:
      static:
        hostnames:
          - tier2-collector-a:4317
          - tier2-collector-b:4317
          - tier2-collector-c:4317

service:
  pipelines:
    traces:
      receivers: [otlp]
      exporters: [loadbalancing]
"""

TIER_2_CONFIG = """
# Tier 2 — tail sampling layer (runs on each of the 3 tier-2 instances)
receivers:
  otlp:
    protocols:
      grpc:
        endpoint: 0.0.0.0:4317

processors:
  tail_sampling:
    decision_wait: 30s
    num_traces: 120000     # 360K total / 3 instances
    expected_new_traces_per_sec: 3333
    policies:
      # ... (same priority stack as Step 8)
      - name: errors
        type: status_code
        status_code: {status_codes: [ERROR]}
      # ... full policies omitted for brevity

exporters:
  otlp:
    endpoint: trace-store:4317
    tls:
      insecure: true

service:
  pipelines:
    traces:
      receivers: [otlp]
      processors: [tail_sampling]
      exporters: [otlp]
"""

# Verify both parse as valid YAML
tier1 = yaml.safe_load(TIER_1_CONFIG)
tier2 = yaml.safe_load(TIER_2_CONFIG)

print("Two-tier topology:")
print("  Tier 1: loadbalancingexporter, routing_key=traceID")
print(f"    Routes to: {len(tier1['exporters']['loadbalancing']['resolver']['static']['hostnames'])} tier-2 instances")
print("  Tier 2: tailsamplingprocessor on each instance")
print(f"    num_traces per instance: {tier2['processors']['tail_sampling']['num_traces']:,}")
print()
print("Why this matters: skip the load balancer and tail sampling silently produces wrong decisions.")
print("Half the spans land on one Collector, half on another, and neither has the complete trace.")
print("Errors fire on whichever Collector got the error span; latency policies miss the late-arriving spans.")


### Step 12: Cost arithmetic at scale

A back-of-envelope at 1M traces/month to illustrate why cost-driven sampling earns its place:

Assumptions:
- 1,000,000 traces/month
- Average trace: 10 spans, 5KB serialized per span = ~50KB/trace
- Trace-store ingestion cost: $0.50 per GB
- Head-only ingestion would cost: 1M × 50KB = 50 GB = $25/mo

That's just ingestion. Storage, query, retention, alerting infra multiply the bill. Real production observability bills run 10-100x the raw ingestion cost at scale.

Apply cost-aware tail sampling: keep 100% of errors + high-cost + enterprise + 5% baseline. From Step 9, that's typically 5-15% retention rate.

In [ ]:
# Cost arithmetic
N_MONTHLY = 1_000_000
AVG_KB_PER_TRACE = 50
INGEST_COST_PER_GB = 0.50

# Head-only ingestion (no sampling)
head_only_gb = (N_MONTHLY * AVG_KB_PER_TRACE) / (1024 * 1024)
head_only_cost = head_only_gb * INGEST_COST_PER_GB

# Cost-aware tail sampling
# Note: in production with realistic traffic distribution (mostly low-cost chat,
# rare high-cost analysis), retention is typically 5-15%. The lab's synthetic
# data has higher avg cost per trace which inflates the high-cost policy match rate.
# We use 12% here to represent a realistic production scenario, not the lab's number.
PRODUCTION_RETENTION_PCT = 12.0
tail_gb = head_only_gb * (PRODUCTION_RETENTION_PCT / 100)
tail_cost = tail_gb * INGEST_COST_PER_GB

print(f"At {N_MONTHLY:,} traces/month, {AVG_KB_PER_TRACE} KB each:")
print()
print("  Head-only ingestion:")
print(f"    Volume: {head_only_gb:>8.2f} GB/mo")
print(f"    Cost:   ${head_only_cost:>8.2f}/mo")
print()
print(f"  Cost-aware tail sampling ({PRODUCTION_RETENTION_PCT:.0f}% retention — typical production):")
print(f"    Volume: {tail_gb:>8.2f} GB/mo")
print(f"    Cost:   ${tail_cost:>8.2f}/mo")
print()
print(f"  Savings: ${head_only_cost - tail_cost:.2f}/mo ({(1 - PRODUCTION_RETENTION_PCT/100) * 100:.0f}% reduction)")
print()
print("The lab's synthetic data shows ~64% retention because the avg cost-per-trace")
print("is artificially high. Real production traffic is dominated by low-cost chat;")
print("the high-cost policy matches a small fraction, hence the typical 5-15% retention.")
print("At larger scales (100M+ traces/month), the multiplier grows because")
print("trace-store query and storage costs are also volume-proportional.")


## Step 13: Synthesis — Path 06 production stack assembled

With Module 6 shipped, the Path 06 production stack covers five operational layers:

| Layer | Module | What it provides | Built on |
|---|---|---|---|
| 1. Instrumentation | M2-3 | Spans with `gen_ai.*` attributes | Application code |
| 2. Online evaluators | M4 | Scores attached to traces | Layer 1 spans |
| 3. Drift detection | M5 | Alerts on score-distribution shifts | Layer 2 scores |
| 4. Calibration | M5 | Trust that scores correlate with humans | Layers 2+3 with a gold set |
| 5. Cost attribution + adaptive sampling | M6 (this) | Per-dimension cost rollups + cost-driven retention | Layer 1 baggage |

Five layers, each enabling the next. Skip any of them and the upper layers either don't work or work without the trust they imply.

**Module 6's specific contributions to the stack**:

- **Identity propagation via baggage** — tenant.id / user.id / task.id flow through every span without explicit argument-passing.
- **The four-token-layer breakdown** — prompt/tool/memory/response, surfaces where spend actually goes.
- **Three attribution dimensions** — per-tenant for unit economics, per-user for cohort analysis, per-task for engineering optimization.
- **The three-layer enforcement ladder** — dashboards, alerts, rate-limit tightening.
- **Cost-driven sampling policies** — `numeric_attribute` on cost; `string_attribute` on tier; probabilistic baseline.
- **The external control loop** — Collector doesn't auto-tune; a sidecar/service does the rate computation and pushes updates.
- **The two-tier Collector topology** — `loadbalancingexporter` + `tailsamplingprocessor` for scale.

**What this lab demonstrated end-to-end**:
- OTel baggage carrying identity from request entry to every downstream span without explicit propagation.
- The four token layers tracked as separate counters per span.
- Per-dimension cost rollup over 200 synthetic traces.
- The "one tenant burning 80% of the budget" signal pattern.
- The 4KB baggage limit and PII/secret discipline.
- A production-realistic Collector YAML with composite policies.
- Cost-driven retention via `numeric_attribute` policy.
- The external `AdaptiveSamplingController` rate-computation loop.
- The two-tier Collector topology for scale.
- Cost arithmetic at 1M traces/mo showing the ~88% cost reduction tail sampling delivers.

**What's deferred**:
- **Module 7 — Multi-turn (threaded) evaluation.** Trajectory metrics across conversation turns. Closes Path 06 v1.
- **Reference solutions for Labs 17-21.** Catchup batch shipping canonical solutions for all five Path 06 labs.
- **Cached-token cost math, FinOps tooling integration, usage forecasting** — all mentioned in concepts pages, not implemented in the lab.

Path 06 v1 is now structurally near-complete with five of seven modules shipped. Module 7 (multi-turn) is the trajectory specialization that closes v1.

✓ **Module 6 complete.**
